# 01 — Exploração dos dados de demanda

Este notebook gera um painel diário sintético de produtos perecíveis e examina cobertura, padrões e rupturas. A coluna `latent_demand` é uma verdade-terreno disponível **somente na simulação**; em dados reais, a demanda durante uma ruptura não é observável apenas pelas vendas.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from demand_forecasting.synthetic import generate_synthetic_demand

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)
data = generate_synthetic_demand(random_state=42)
data.head()

## Estrutura e cobertura

Verifique tipos, nulos e se cada produto possui observações diárias no período gerado antes de modelar.

In [ ]:
print(f"Formato: {data.shape}; produtos: {data['product_id'].nunique()}")
print(f"Período: {data['date'].min().date()} a {data['date'].max().date()}")
display(data.dtypes.to_frame("dtype"))
display(data.isna().sum().to_frame("nulos"))
coverage = data.groupby("product_id").agg(
    dias=("date", "nunique"),
    primeira_data=("date", "min"),
    ultima_data=("date", "max"),
)
display(coverage)

## Vendas por produto e padrão semanal

Os gráficos ajudam a identificar diferenças de escala e sazonalidade. As séries foram simuladas para fins didáticos e não representam uma operação comercial.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), constrained_layout=True)
sns.lineplot(data=data, x="date", y="sales", hue="product_id",
             estimator=None, alpha=0.75, ax=axes[0])
axes[0].set(title="Vendas diárias observadas por produto", ylabel="Unidades")
weekday = (data.assign(dia_semana=data["date"].dt.day_name())
           .groupby(["product_id", "dia_semana"], as_index=False)["sales"].mean())
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday",
                 "Friday", "Saturday", "Sunday"]
sns.lineplot(data=weekday, x="dia_semana", y="sales", hue="product_id",
             marker="o", ax=axes[1])
axes[1].set(title="Média de vendas por dia da semana", ylabel="Unidades")
axes[1].tick_params(axis="x", rotation=25)
plt.show()

## Disponibilidade e censura das vendas

Em uma ruptura, vendas registradas podem ficar abaixo da demanda. `latent_demand` permite ilustrar essa diferença no **dataset simulado**, mas não deve ser usada como atributo ou rótulo em um cenário real. Aqui, `stockout` também é um sinal sintético perfeito; sistemas reais exigem dados confiáveis de estoque/disponibilidade.

In [ ]:
stockout_summary = data.groupby("product_id").agg(
    dias=("date", "size"),
    dias_com_ruptura=("stockout", "sum"),
    proporcao_ruptura=("stockout", "mean"),
    vendas_media=("sales", "mean"),
    demanda_latente_media_simulada=("latent_demand", "mean"),
)
stockout_summary["unidades_censuradas_simuladas"] = (
    data["latent_demand"] - data["sales"]
).groupby(data["product_id"]).sum()
display(stockout_summary.round(3))

fig, ax = plt.subplots(figsize=(9, 4))
sample = data.loc[data["product_id"] == data["product_id"].iloc[0]]
ax.plot(sample["date"], sample["sales"], label="Vendas observadas", alpha=0.8)
ax.plot(sample["date"], sample["latent_demand"], label="Demanda latente (simulada)", alpha=0.7)
ax.set(title=f"Vendas versus demanda sintética — {sample['product_id'].iloc[0]}", ylabel="Unidades")
ax.legend()
plt.show()

## Conclusões e próximos passos

- As séries diferem em nível e apresentam padrões semanais e sazonais.
- Rupturas censuram a venda observada; ao treinar, os exemplos sintéticos sinalizados como ruptura serão excluídos.
- No próximo notebook, criaremos atributos conhecidos na origem da previsão e compararemos baselines com Random Forest por backtesting temporal.
- Não interprete métricas deste conjunto sintético como desempenho esperado em produção.